In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, inspect
from IPython.display import display
import json

# 1. COMANDOS PARA RECARGA AUTOMÁTICA
%load_ext autoreload
%autoreload 2
%matplotlib inline

pd.set_option('display.max_columns', None)   # mostrar todas las columnas
pd.set_option('display.width', 0)           # dejar que use todo el ancho disponible
pd.set_option('display.max_colwidth', None) # Quitar el límite de ancho de las columnas
pd.set_option('display.expand_frame_repr', False) # Para que no "envuelva" la tabla y se mantenga en una sola fila larga

# Descarga de bases de datos:

In [ ]:
def load_and_process_journal(db_relative_path="../.data/journal.db", table_name=None):
    """
    Conecta a la DB, extrae los datos y aplana la columna 'payload' de JSON a columnas.
    """
    # 1. Conexión
    engine = create_engine(f"sqlite:///{db_relative_path}")
    inspector = inspect(engine)
    tablas = inspector.get_table_names()
    
    if not tablas:
        print("❌ La base de datos está vacía o no se encuentra en la ruta especificada.")
        return None

    # 2. Selección de tabla
    target_table = table_name if table_name else tablas[0]
    print(f"✅ Tablas detectadas: {tablas}")
    print(f"📊 Procesando tabla: '{target_table}'")

    # 3. Carga de datos crudos
    df_raw = pd.read_sql(f"SELECT * FROM {target_table}", engine)
    
    if 'payload' not in df_raw.columns:
        print(f"⚠️ La tabla '{target_table}' no contiene una columna 'payload'.")
        return df_raw

    # 4. Procesamiento de JSON
    def parse_payload(x):
        try:
            return json.loads(x) if isinstance(x, str) else x
        except (json.JSONDecodeError, TypeError):
            return {}

    # Aplanamos el JSON
    df_payload = pd.json_normalize(df_raw['payload'].apply(parse_payload))
    
    # Combinamos con el resto de columnas y eliminamos el 'payload' original
    df_final = pd.concat([df_raw.drop(columns=['payload']), df_payload], axis=1)

    # 5. Reporte y Visualización
    print(f"✨ Proceso completado: {df_final.shape[0]} registros y {df_final.shape[1]} columnas.")
    display(df_final.head())
    
    return df_final

In [ ]:
# --- USO DE LA FUNCIÓN ---
df = load_and_process_journal()